# Explore prepared RDD2022
Run the preparation CLI first. Use this notebook for inspecting its outputs, not producing benchmark results.

In [ ]:
import csv
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
prepared = repo / "data/processed/rdd_yolo"
with (prepared / "stats.csv").open(newline="", encoding="utf-8") as stream:
    stats = list(csv.DictReader(stream))
for row in stats:
    print(row["domain"], row["split"], row["images"], "images;",
          {code: row[code] for code in ("D00", "D10", "D20", "D40")})


In [ ]:
with (prepared / "manifest.csv").open(newline="", encoding="utf-8") as stream:
    samples = list(csv.DictReader(stream))

# Change this selection to inspect each country and class.
row = next(row for row in samples if int(row["boxes"]) > 0)
with Image.open(prepared / row["image"]) as source:
    canvas = source.convert("RGB")
draw = ImageDraw.Draw(canvas)
w, h = canvas.size
codes = ("D00", "D10", "D20", "D40")
for line in (prepared / row["label"]).read_text().splitlines():
    category, x, y, bw, bh = line.split()
    x, y, bw, bh = map(float, (x, y, bw, bh))
    left, top = (x - bw / 2) * w, (y - bh / 2) * h
    draw.rectangle((left, top, (x + bw / 2) * w, (y + bh / 2) * h),
                   outline="yellow", width=3)
    draw.text((left, top), codes[int(category)], fill="yellow", stroke_width=1)
canvas.thumbnail((900, 700))
print(row["source_image"], row["split"])
display(canvas)
